In [19]:
import pandas as pd

df = pd.read_csv("journal_allowlist.csv")

df["qualified"].value_counts()

qualified
False    12198
True      7235
Name: count, dtype: int64

In [21]:
# 1. Latin/Greek Stems (Catches English, Spanish, Portuguese, and German simultaneously)
shared_stems = [
    # Medical Specialties
    "cardi", "derma", "gastro", "hemato", "immuno", "oncol", "ophthal", "optom", 
    "ortho", "osteo", "pediatr", "psiquiatr", "psychiat", "radiol", "toxicol", 
    "urol", "virol", "bacteriol", "parasitol", "epidemiol", "pathol", "patologia", 
    "pharm", "farmac", "nutri", "endocrin", "obstet", "gynecol", "ginecol",
    
    # Procedures / Care / Biology
    "surg", "cirug", "cirurg", "chirurg", "nurs", "enferm", "pflege", "dent", 
    "odont", "zahn", "veterin", "tierarzt", "anatomi", "physiol", "fisiol",
    
    # Hard Sciences / Engineering / Earth
    "astron", "aerosp", "agricult", "agronom", "botan", "zoolog", "ecolog", 
    "geolog", "metallurg", "mechanic", "mecanic", "thermodynam", "ingenier", 
    "engenhar", "ingenieur", "architect", "arquitec", "mathematic", "matematic",
    
    # Business & Law
    "account", "contabil", "finance", "finanz", "finanç", "marketing", "supply chain", 
    "logistics", "juris", "juridi"
]

# 2. German Specific Non-Latin Roots
german_specific = [
    "arzt", "ärzte", "heilkunde", "kranken", "apotheke", "recht", "wirtschaft",
    "bauwesen", "umwelt", "betriebs", "steuer"
]

# 3. Spanish/Portuguese Specific Non-Latin Roots
es_pt_specific = [
    "salud", "saúde", "derecho", "direito", "abogado", "advogado", "negocio",
    "empresa", "medio ambiente", "meio ambiente"
]

# 4. Indonesian (Bahasa Indonesia)
indonesian_specific = [
    "kedokteran", "medis", "klinis", "bedah", "pasien", "apotek", "perawat", 
    "gigi", "hewan", "kebidanan", "gizi", "teknik", "pertanian", "peternakan", 
    "hukum", "ekonomi", "akuntansi", "manajemen", "pajak", "kehutanan"
]

# 5. Chinese (中文) - Grouped by fields
chinese_specific = [
    # Medical & Clinical
    "医学", "临床", "外科", "内科", "药", "护理", "病理", "毒理", "肿瘤", "牙科", 
    "口腔", "兽医", "心血管", "胃肠", "皮肤", "免疫", "妇产", "儿科", "精神",
    # Hard Sciences & Engineering
    "物理", "化学", "工程", "材料", "建筑", "航空", "航天", "机械", "电气", "计算",
    # Environment & Business
    "农业", "林业", "法律", "经济", "金融", "会计", "管理", "商业"
]

# Combine all lists
all_excludes = shared_stems + german_specific + es_pt_specific + indonesian_specific + chinese_specific

# Join with pipe for regex
exclude_pattern = "|".join(all_excludes)

# Normalize the journal column to lowercase and remove accents for the Latin-script matches
# (This ensures "Cirugía" matches "cirug")
df['journal_clean'] = df['journal'].str.lower().str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

# Apply the filter
df.loc[df["journal_clean"].str.contains(exclude_pattern, na=False), "qualified"] = False

# Drop the temporary clean column if you no longer need it
df = df.drop(columns=['journal_clean'])

In [22]:
df["qualified"].value_counts()

qualified
False    13188
True      6245
Name: count, dtype: int64

In [23]:
df.to_csv("journal_allowlist.csv")